# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR² (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
> Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
Load the Croissant metadata and records from the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors: {[author['@id'] if isinstance(author, dict) and '@id' in author else author for author in getattr(metadata, 'author', [])]}")
print(f"Version: {metadata.version if hasattr(metadata,'version') else 'N/A'}")

## 2. Data Overview
Let's review the available record sets (tables), fields, and their `@id`s.
Below, we will programmatically inspect available record sets and their fields by `@id` for reference in future steps.

In [ ]:
# List available record sets and their fields by `@id`
from pprint import pprint

record_sets = [r for r in getattr(metadata, 'record_sets', [])] if hasattr(metadata, 'record_sets') else []
# The property might also be 'recordSet' (singular)
if not record_sets and hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
# Support both dict or list-of-dict: Check for @id and fields
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        # rs may be a dict or just an @id string
        if isinstance(rs, dict):
            rs_id = rs.get('@id', '(no id found)')
            print(f"RecordSet @id: {rs_id}")
            fields = rs.get('field', [])
            if not isinstance(fields, list):
                fields = [fields]
            print("  Fields by @id:")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id','(no id)')}")
                else:
                    print(f"    - {field}")
            print()
        else:
            print(f"RecordSet @id: {rs}")

#### Alternatively, let's enumerate all record set `@id`s that can be used with mlcroissant
<br/>
*We will also demonstrate how to print the fields (`@id`s) of each record set in a machine-readable way for use in later extraction steps.*

In [ ]:
# Helper: List record set IDs and field IDs properly (for use with mlcroissant)
def get_croissant_recordset_ids_and_field_ids(ds_metadata):
    # Try both 'record_sets' / 'recordSet' to be robust to Croissant structure
    record_sets = getattr(ds_metadata, 'record_sets', None)
    if record_sets is None:
        record_sets = getattr(ds_metadata, 'recordSet', [])
    
    ids_fields = []
    for rs in record_sets:
        rs_id = rs.get('@id') if isinstance(rs, dict) and '@id' in rs else rs
        # fields can be a list of @id or dicts or just one field
        fs = rs.get('field', []) if isinstance(rs, dict) else []
        if not isinstance(fs, list):
            fs = [fs]
        # Convert dict fields to @id if needed
        field_ids = []
        for field in fs:
            if isinstance(field, dict):
                field_ids.append(field.get('@id', field))
            else:
                field_ids.append(field)
        ids_fields.append({'record_set_id': rs_id, 'field_ids': field_ids})
    return ids_fields

ids_fields = get_croissant_recordset_ids_and_field_ids(metadata)
if not ids_fields:
    print("No record sets w/ fields detected.")
else:
    for rs in ids_fields:
        print(f"RecordSet @id: {rs['record_set_id']}")
        print(f"  Field @id's: {rs['field_ids']}")
        print()

##### If there are no record sets in the schema (they may be empty), let's try to list record sets available dynamically from the dataset instance for robust demonstration.

In [ ]:
# For demonstration, list record set @id's as seen directly available to mlcroissant
all_rs_ids = dataset.record_sets
print("Available record sets in the schema (mlcroissant-detected):")
for rs_id in all_rs_ids:
    print(f"- {rs_id}")

## 3. Data Extraction
Here we will extract records from a record set of interest (using its `@id`).

Because Croissant schemas may have only one main tabular record set, we attempt to use the first detected record set for demonstration.

In [ ]:
# Extract ALL record sets into dataframes using their @id
import itertools

record_set_ids = list(dataset.record_sets)
dataframes = {}
for rs_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=rs_id)
        # Support both non-empty and empty
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set {rs_id}: shape {df.shape}")
    except Exception as e:
        print(f"Could not load {rs_id}: {e}")

# Show columns for the first available record set dataframe
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nMain record set @id used for EDA: {main_rs_id}")
    print(f"Columns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head(5))
else:
    print("No record set dataframes could be created.")

## 4. Exploratory Data Analysis (EDA)
We now select numeric, categorical, and group fields by their `@id` (and corresponding DataFrame columns).

*The example below demonstrates filtering, normalization, and grouping. Please replace the variables if your analysis requires different fields based on the actual data loaded above.*

In [ ]:
# Please review the column list above to select field @id's for analysis.
# Below, set the numeric_field and group_field variables to appropriate @id's/column names for your EDA.

# Set these based on printed columns. Example field ids (replace as appropriate):
# numeric_field_id = '@id_for_numeric_variable'
# group_field_id = '@id_for_categorical_variable'

if dataframes:
    df = dataframes[main_rs_id]
    
    # For demonstration, try to automatically pick a numeric field
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    # Or fallback to common names (may not match your particular dataset)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        numeric_field = None

    print(f"Numeric field selected for analysis: {numeric_field}")

    # Try to pick a group/categorical field
    group_candidates = [col for col in df.columns if df[col].dtype == object]
    group_field = group_candidates[0] if group_candidates else None
    print(f"Group/categorical field: {group_field}")

    if numeric_field:
        # Example: Threshold as 10 for demonstration
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group and aggregate
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA in main record set.")
else:
    print("No record sets to analyze.")

## 5. Visualization
Visualize distributions or relationships between fields of interest.

The example below visualizes the distribution of the selected numeric variable and, if applicable, a grouped bar plot by the categorical/group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and numeric_field:
    # Numeric distribution
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, color='skyblue', kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Grouped bar (if group_field exists and is not high cardinality)
    if group_field and df[group_field].nunique() < 10:
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("No fields suitable for visualization.")

## 6. Conclusion

- We loaded metadata and records for the FAIR² tabular dataset using its Croissant schema and the `mlcroissant` library.
- We programmatically explored record set and field @id structure.
- Example data processing and basic visualizations were performed.
- For specific clinical or research questions, further field/code customization would be based on the dataset's columns and schema.

---
<sub>All entities referenced used their `@id` throughout to ensure traceability to the Croissant schema.</sub>